# PaySim DL Model Training

Dataset: PaySim Fraud Detection (Engineered Features, Scaled)

| ModelID | Model Type | Notes |
|--------|-----------|------|
| AE_Semi | Autoencoder | Semi-supervised regime, trained on normal transactions only, anomaly score from MSE reconstruction error |
| AE_Supp | Autoencoder | Supervised-style regime, trained on all data, anomaly score from MSE reconstruction error |
| MLP | Multi-Layer Perceptron | Supervised classification, weighted BCE loss to handle class imbalance, 128-64-1 architecture |

In [1]:
import os
import pandas as pd
import numpy as np
import gc
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

import warnings
warnings.filterwarnings('ignore')

os.makedirs('results', exist_ok=True)

# --- 1. DATA PREP ---
print("Loading and scaling data...")
df = pd.read_parquet('data/paysim_feature.parquet')

split_idx = int(len(df) * 0.8)

X = df.drop(['isFraud'], axis=1, errors='ignore').fillna(-999)
y = df['isFraud'].values

X_train_raw, X_val_raw = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_val = scaler.transform(X_val_raw).astype(np.float32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
del df, X, X_train_raw, X_val_raw
gc.collect()

# --- 2. THE AUTOENCODER REGIMES ---
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 16)
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

def train_ae(train_data, input_dim, device, epochs=10):
    model = Autoencoder(input_dim).to(device)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(train_data)),
        batch_size=2048,
        shuffle=True
    )
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        for batch in loader:
            xb = batch[0].to(device)
            loss = loss_fn(model(xb), xb)
            opt.zero_grad()
            loss.backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        val_t = torch.from_numpy(X_val).to(device)
        recon = model(val_t)
        scores = torch.mean((val_t - recon) ** 2, dim=1).cpu().numpy()

    return scores

# Regime A: Semi-supervised (Train on NORMAL only)
print(">>> Training AE: Semi-supervised Regime...")
ae_semi_scores = train_ae(X_train[y_train == 0], X_train.shape[1], device)

# Regime B: Supervised-style AE (Train on ALL data)
print(">>> Training AE: Supervised Regime...")
ae_supp_scores = train_ae(X_train, X_train.shape[1], device)

# --- 3. SUPERVISED MLP ---
print("\n>>> Training Supervised MLP...")
print("Handling imbalance via weighted loss...")

pos_weight = torch.tensor([len(y_train[y_train == 0]) / len(y_train[y_train == 1])]).to(device)

mlp_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(X_train),
        torch.from_numpy(y_train.astype(np.float32))
    ),
    batch_size=2048,
    shuffle=True
)

class SupervisedMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

mlp = SupervisedMLP(X_train.shape[1]).to(device)
mlp_opt = torch.optim.Adam(mlp.parameters(), lr=1e-3)
mlp_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

for epoch in range(10):
    mlp.train()
    for xb, yb in mlp_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = mlp(xb).squeeze()
        loss = mlp_criterion(logits, yb)

        mlp_opt.zero_grad()
        loss.backward()
        mlp_opt.step()

    print(f"MLP Epoch {epoch+1}/10 complete")

mlp.eval()
with torch.no_grad():
    val_tensor = torch.from_numpy(X_val).to(device)
    mlp_probs = torch.sigmoid(mlp(val_tensor)).cpu().numpy().flatten()

# --- 4. EVALUATION SUMMARY ---
summary_rows = []

for model_name, scores in {
    'ae_semi_supervised': ae_semi_scores,
    'ae_supervised': ae_supp_scores,
    'mlp_prob': mlp_probs
}.items():
    roc = roc_auc_score(y_val, scores)
    auprc = average_precision_score(y_val, scores)
    summary_rows.append({
        'Model': model_name,
        'ROC-AUC': roc,
        'AUPRC': auprc
    })

summary_df = pd.DataFrame(summary_rows).sort_values('AUPRC', ascending=False)

print("\n--- PAYSIM DL PERFORMANCE RANKING ---")
print(summary_df)

# --- 5. EXPORT ---
results_dict = {
    'actual': y_val,
    'ae_semi_supervised': ae_semi_scores,
    'ae_supervised': ae_supp_scores,
    'mlp_prob': mlp_probs
}

df_final = pd.DataFrame(results_dict)
df_final.to_csv('results/paysim_dl_results.csv', index=False)

print("\nSuccess! Results saved to 'results/paysim_dl_results.csv' with all regimes.")

Loading and scaling data...
>>> Training AE: Semi-supervised Regime...
>>> Training AE: Supervised Regime...

>>> Training Supervised MLP...
Handling imbalance via weighted loss...
MLP Epoch 1/10 complete
MLP Epoch 2/10 complete
MLP Epoch 3/10 complete
MLP Epoch 4/10 complete
MLP Epoch 5/10 complete
MLP Epoch 6/10 complete
MLP Epoch 7/10 complete
MLP Epoch 8/10 complete
MLP Epoch 9/10 complete
MLP Epoch 10/10 complete

--- PAYSIM DL PERFORMANCE RANKING ---
                Model   ROC-AUC     AUPRC
2            mlp_prob  0.997747  0.886387
0  ae_semi_supervised  0.875687  0.237070
1       ae_supervised  0.880911  0.207831

Success! Results saved to 'results/paysim_dl_results.csv' with all regimes.
